# NNN-slug — training

Template training notebook. Copy into `experiments/NNN-slug/train.ipynb`.

The cell below is tagged `parameters` so papermill can override any value:

```
papermill train.ipynb artifacts/out.ipynb -p lr 3e-4 -p backbone densenet121
```

The notebook **must** end with `log_run(...)`. A run that is not logged did not happen.

Live curves: `tensorboard --logdir experiments` from `src/`.

In [ ]:
# Every value papermill may override. Keep this cell minimal and flat.
exp        = "000-template"
backbone   = "resnet18"
lr         = 1e-4
epochs     = 2
batch_size = 16
dataset    = "synthetic"
device     = "mps"    # AMD Radeon Pro 5300M via Metal; "cpu" to compare
seed       = 42
note       = ""

In [ ]:
import os

import matplotlib.pyplot as plt
import torch
import torch.nn as nn

from uavloc.track import log_history, log_run, set_seed, tensorboard_writer

set_seed(seed)
writer = tensorboard_writer(exp)
# MPS works on this Intel Mac via the AMD Radeon Pro 5300M (~2.3x faster
# than CPU on a resnet18 training step). Op coverage has gaps; the fallback
# sends anything unsupported to CPU rather than raising.
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

In [ ]:
# --- replace this block with real data + model -------------------------------
X = torch.randn(128, 3, 32, 32, device=device)
y = torch.randint(0, 2, (128,), device=device)
model = nn.Sequential(nn.Flatten(), nn.Linear(3 * 32 * 32, 2)).to(device)
opt = torch.optim.Adam(model.parameters(), lr=lr)
lossf = nn.CrossEntropyLoss()

train_hist, val_hist = [], []
for ep in range(epochs):
    model.train()
    opt.zero_grad()
    loss = lossf(model(X[:96]), y[:96])
    loss.backward()
    opt.step()
    with torch.no_grad():
        val = lossf(model(X[96:]), y[96:]).item()
    train_hist.append(loss.item())
    val_hist.append(val)
    log_history(writer, {"step": ep + 1, "train_loss": loss.item(), "val_loss": val})
    print(f"epoch {ep+1}/{epochs}  train {loss.item():.4f}  val {val:.4f}")
# -----------------------------------------------------------------------------

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(train_hist, label="train")
ax.plot(val_hist, label="val")
ax.set_xlabel("epoch"); ax.set_ylabel("loss"); ax.legend()
fig.tight_layout()

In [ ]:
log_run(
    exp=exp,
    config={"backbone": backbone, "lr": lr, "epochs": epochs,
            "batch_size": batch_size, "dataset": dataset},
    metrics={"best_val": min(val_hist), "final_train": train_hist[-1],
             "val_loss": val_hist},
    note=note,
    figures=[fig],
    seed=seed,
    device=device,
    writer=writer,
)